In [2]:
# Raster Preparation for Final Models 
# Refer to notes for all variables and resampling requirement

import os
import gc
import glob
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
from rasterio.enums import Resampling
from pathlib import Path

# Configuration 
START = pd.Timestamp("2001-01-01")
END   = pd.Timestamp("2005-12-31")
 
WET_DAY_THRESHOLD = 0.1   # mm — gridded wet-day definition
 
DIRS = {
    "era5l_prec":     r"..\Data\Datasets\ERA5L_prec",
    "chirps_prec":    r"..\Data\Datasets\CHIRPS_prec",
    "har_prec":       r"..\Data\Datasets\HAR_prec",
    "persiann_prec":  r"..\Data\Datasets\PERSIANN_prec",
    "mswep_prec":     r"..\Data\Datasets\MSWEP_prec",
    "era5l_k_prec":   r"..\Data\Processed\Kriging_Predictors\Kriged_rasters",
    "era5l_wind":     r"..\Data\Datasets\ERA5L_2_wind_speed",
    "era5l_evap":     r"..\Data\Datasets\ERA5L_2_total_evaporation",
    "era5l_pressure": r"..\Data\Datasets\ERA5L_2_surface_pressure",
    "era5l_soil":     r"..\Data\Datasets\ERA5L_2_soil_moisture",
    "era5l_dewpoint": r"..\Data\Datasets\ERA5L_2_dewpoint_temperature",
    "modis_evi":      r"..\Data\Datasets\MODIS_2_EVI"
}
 
OUTPUT_DIR   = r"..\Data\Processed\RegressionModel\Monthly_Rasters"
TEMPLATE_PATH = r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_max.tif"
 
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Template ────────────────────────────────────────────────────────────────── 
template = rxr.open_rasterio(TEMPLATE_PATH, masked=True) 

In [3]:
# ── Helpers ───────────────────────────────────────────────────────────────────
 
def resample_to_template(raster: xr.DataArray) -> xr.DataArray:
    """Reproject and clip raster to match the 1km WGS84 template."""
    return raster.rio.reproject_match(template, resampling=Resampling.bilinear)
 
 
def parse_tif_catalog(directory: str) -> pd.DataFrame:
    """
    Build a DataFrame of (filepath, date) for all .tif files in a directory.
    Assumes filenames end with YYYYMMDD.tif (last 8 chars before extension).
    Returns sorted by date.
    """
    files = sorted(glob.glob(os.path.join(directory, "*.tif")))
    if not files:
        raise FileNotFoundError(f"No .tif files found in {directory}")
    rows = []
    for f in files:
        stem = Path(f).stem          # filename without extension
        date_str = stem[-8:]         # last 8 chars = YYYYMMDD
        try:
            date = pd.Timestamp(date_str)
            rows.append({"path": f, "date": date})
        except ValueError:
            continue
    return pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
 
 
def filter_period(catalog: pd.DataFrame, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    return catalog[(catalog["date"] >= start) & (catalog["date"] <= end)].reset_index(drop=True)
 
 
def stack_month(catalog: pd.DataFrame, year: int, month: int) -> xr.DataArray | None:
    """
    Load all daily rasters for year/month, resample each, and stack along
    a new 'day' dimension. Returns None if no files found for that month.
    Memory note: opens one file at a time, closes after stacking.
    """
    month_files = catalog[
        (catalog["date"].dt.year == year) &
        (catalog["date"].dt.month == month)
    ]
    if month_files.empty:
        return None
 
    slices = []
    for _, row in month_files.iterrows():
        da = rxr.open_rasterio(row["path"], masked=True)
        da = resample_to_template(da)
        # Squeeze band dim if single-band
        if "band" in da.dims and da.sizes["band"] == 1:
            da = da.squeeze("band", drop=True)
        da = da.assign_coords(day=row["date"])
        slices.append(da.load())  # load into memory before closing
        da.close()
 
    stack = xr.concat(slices, dim="day")
    return stack
 
 
def save_raster(data: xr.DataArray, out_path: str, crs, nodata: float = -9999.0):
    """Write a 2D DataArray to GeoTiff, preserving CRS and nodata."""
    data = data.rio.write_crs(crs)
    data = data.rio.write_nodata(nodata)
    data.rio.to_raster(out_path)
 
 
def month_label(year: int, month: int) -> str:
    return f"{year}{month:02d}"
 
 
# Build list of (year, month) to process
month_range = pd.period_range(start=START, end=END, freq="M")

In [4]:
# ── 1. Precipitation products ─────────────────────────────────────────────────
#    Aggregates: monthly_sum (all products)
#                wet_day_count, monthly_std (ERA5L only)
 
PREC_PRODUCTS = {
    "era5l":    DIRS["era5l_prec"],
    "chirps":   DIRS["chirps_prec"],
    "har":      DIRS["har_prec"],
    "persiann": DIRS["persiann_prec"],
    'mswep': DIRS['mswep_prec'],
    'era5l_k': DIRS['era5l_k_prec']
}
 
print("=" * 60)
print("Processing precipitation products...")
print("=" * 60)
 
for product, directory in PREC_PRODUCTS.items():
    product_out = os.path.join(OUTPUT_DIR, product)
    os.makedirs(product_out, exist_ok=True)
 
    catalog = filter_period(parse_tif_catalog(directory), START, END)
    crs = rxr.open_rasterio(catalog["path"].iloc[0]).rio.crs
 
    for period in month_range:
        year, month = period.year, period.month
        label = month_label(year, month)
 
        sum_path = os.path.join(product_out, f"{product}_monthly_sum_{label}.tif")
        # Skip if all outputs for this product/month already exist
        extra_exist = True
        if product == "era5l":
            wdc_path = os.path.join(product_out, f"era5l_wet_day_count_{label}.tif")
            std_path = os.path.join(product_out, f"era5l_monthly_std_{label}.tif")
            extra_exist = os.path.exists(wdc_path) and os.path.exists(std_path)
        if os.path.exists(sum_path) and extra_exist:
            print(f"  [{product}] {label} — skipping (exists)")
            continue
 
        stack = stack_month(catalog, year, month)
        if stack is None:
            print(f"  [{product}] {label} — WARNING: no files found, skipping")
            continue
 
        # Monthly sum
        monthly_sum = stack.sum(dim="day", skipna=True)
        save_raster(monthly_sum, sum_path, crs)
 
        # ERA5L-only additional aggregates
        if product == "era5l":
            wet_day_count = (stack >= WET_DAY_THRESHOLD).sum(dim="day")
            wet_day_count = wet_day_count.astype(np.float32)
            save_raster(wet_day_count, wdc_path, crs)
 
            monthly_std = stack.std(dim="day", skipna=True)
            save_raster(monthly_std, std_path, crs)
 
        del stack
        gc.collect()
        print(f"  [{product}] {label} — done")

Processing precipitation products...
  [era5l] 200101 — skipping (exists)
  [era5l] 200102 — skipping (exists)
  [era5l] 200103 — skipping (exists)
  [era5l] 200104 — skipping (exists)
  [era5l] 200105 — skipping (exists)
  [era5l] 200106 — skipping (exists)
  [era5l] 200107 — skipping (exists)
  [era5l] 200108 — skipping (exists)
  [era5l] 200109 — skipping (exists)
  [era5l] 200110 — skipping (exists)
  [era5l] 200111 — skipping (exists)
  [era5l] 200112 — skipping (exists)
  [era5l] 200201 — skipping (exists)
  [era5l] 200202 — skipping (exists)
  [era5l] 200203 — skipping (exists)
  [era5l] 200204 — skipping (exists)
  [era5l] 200205 — skipping (exists)
  [era5l] 200206 — skipping (exists)
  [era5l] 200207 — skipping (exists)
  [era5l] 200208 — skipping (exists)
  [era5l] 200209 — skipping (exists)
  [era5l] 200210 — skipping (exists)
  [era5l] 200211 — skipping (exists)
  [era5l] 200212 — skipping (exists)
  [era5l] 200301 — skipping (exists)
  [era5l] 200302 — skipping (exists)
 

In [5]:
# ── 2. ERA5L meteorological variables ─────────────────────────────────────────
#    All: monthly_mean
#    total_evaporation: also monthly_sum (physically a flux accumulation)
 
MET_PRODUCTS = {
    "era5l_wind":     (DIRS["era5l_wind"],     "mean"),
    "era5l_evap":     (DIRS["era5l_evap"],     "both"),   # mean + sum
    "era5l_pressure": (DIRS["era5l_pressure"], "mean"),
    "era5l_soil":     (DIRS["era5l_soil"],     "mean"),
    "era5l_dewpoint": (DIRS["era5l_dewpoint"], "mean"),
}
 
print("\n" + "=" * 60)
print("Processing ERA5L meteorological variables...")
print("=" * 60)
 
for product, (directory, agg_type) in MET_PRODUCTS.items():
    product_out = os.path.join(OUTPUT_DIR, product)
    os.makedirs(product_out, exist_ok=True)
 
    catalog = filter_period(parse_tif_catalog(directory), START, END)
    crs = rxr.open_rasterio(catalog["path"].iloc[0]).rio.crs
 
    for period in month_range:
        year, month = period.year, period.month
        label = month_label(year, month)
 
        mean_path = os.path.join(product_out, f"{product}_monthly_mean_{label}.tif")
        if os.path.exists(mean_path):
            print(f"  [{product}] {label} — skipping (exists)")
            continue
 
        stack = stack_month(catalog, year, month)
        if stack is None:
            print(f"  [{product}] {label} — WARNING: no files found, skipping")
            continue
 
        monthly_mean = stack.mean(dim="day", skipna=True)
        save_raster(monthly_mean, mean_path, crs)
 
        if agg_type == "both":
            sum_path = os.path.join(product_out, f"{product}_monthly_sum_{label}.tif")
            monthly_sum = stack.sum(dim="day", skipna=True)
            save_raster(monthly_sum, sum_path, crs)
 
        del stack
        gc.collect()
        print(f"  [{product}] {label} — done")


Processing ERA5L meteorological variables...
  [era5l_wind] 200101 — skipping (exists)
  [era5l_wind] 200102 — skipping (exists)
  [era5l_wind] 200103 — skipping (exists)
  [era5l_wind] 200104 — skipping (exists)
  [era5l_wind] 200105 — skipping (exists)
  [era5l_wind] 200106 — skipping (exists)
  [era5l_wind] 200107 — skipping (exists)
  [era5l_wind] 200108 — skipping (exists)
  [era5l_wind] 200109 — skipping (exists)
  [era5l_wind] 200110 — skipping (exists)
  [era5l_wind] 200111 — skipping (exists)
  [era5l_wind] 200112 — skipping (exists)
  [era5l_wind] 200201 — skipping (exists)
  [era5l_wind] 200202 — skipping (exists)
  [era5l_wind] 200203 — skipping (exists)
  [era5l_wind] 200204 — skipping (exists)
  [era5l_wind] 200205 — skipping (exists)
  [era5l_wind] 200206 — skipping (exists)
  [era5l_wind] 200207 — skipping (exists)
  [era5l_wind] 200208 — skipping (exists)
  [era5l_wind] 200209 — skipping (exists)
  [era5l_wind] 200210 — skipping (exists)
  [era5l_wind] 200211 — skippi

In [5]:
# ── 3. MODIS EVI (16-day composites) ─────────────────────────────────────────
#    Monthly mean of composites overlapping each month.
#    Missing fraction = 1 - (available composites / expected composites).
#    No daily interpolation — directly average overlapping 16-day windows.
#
#    A composite with start date D covers [D, D+15].
#    Include composites where: D <= month_end AND D+15 >= month_start
 
print("\n" + "=" * 60)
print("Processing MODIS EVI (16-day composites → monthly mean)...")
print("=" * 60)
 
evi_out = os.path.join(OUTPUT_DIR, "modis_evi")
os.makedirs(evi_out, exist_ok=True)
 
evi_catalog = parse_tif_catalog(DIRS["modis_evi"])
# No period filter here — we need composites that overlap the period
# including boundary composites
evi_catalog["end_date"] = evi_catalog["date"] + pd.Timedelta(days=15)
 
# Estimate expected composites per month (16-day → ~2 per month)
def expected_composites(year: int, month: int) -> int:
    """Approximate number of 16-day composites expected to overlap a month."""
    days_in_month = pd.Period(f"{year}-{month:02d}", freq="M").days_in_month
    return max(1, round(days_in_month / 16))
 
crs_evi = rxr.open_rasterio(evi_catalog["path"].iloc[0]).rio.crs
 
for period in month_range:
    year, month = period.year, period.month
    label = month_label(year, month)
 
    mean_path    = os.path.join(evi_out, f"evi_monthly_mean_{label}.tif")
    missing_path = os.path.join(evi_out, f"evi_missing_fraction_{label}.tif")
 
    if os.path.exists(mean_path) and os.path.exists(missing_path):
        print(f"  [EVI] {label} — skipping (exists)")
        continue
 
    month_start = pd.Timestamp(year=year, month=month, day=1)
    month_end   = month_start + pd.offsets.MonthEnd(0)
 
    # Composites that overlap this month
    overlapping = evi_catalog[
        (evi_catalog["date"]     <= month_end) &
        (evi_catalog["end_date"] >= month_start)
    ]
 
    n_expected  = expected_composites(year, month)
    n_available = len(overlapping)
    missing_frac = max(0.0, (n_expected - n_available) / n_expected)
 
    if n_available == 0:
        print(f"  [EVI] {label} — WARNING: no composites found, skipping")
        continue
 
    # Load, resample, and average overlapping composites
    slices = []
    for _, row in overlapping.iterrows():
        da = rxr.open_rasterio(row["path"], masked=True)
        da = resample_to_template(da)
        if "band" in da.dims and da.sizes["band"] == 1:
            da = da.squeeze("band", drop=True)
        slices.append(da.load())
        da.close()
 
    stack   = xr.concat(slices, dim="composite")
    evi_mean = stack.mean(dim="composite", skipna=True)
    save_raster(evi_mean, mean_path, crs_evi)
 
    # Missing fraction raster — spatially constant scalar stored as raster
    missing_arr = xr.full_like(evi_mean, fill_value=float(missing_frac))
    save_raster(missing_arr, missing_path, crs_evi, nodata=-9999.0)
 
    del stack, slices
    gc.collect()
    print(f"  [EVI] {label} — {n_available}/{n_expected} composites, missing={missing_frac:.2f}")
 
print("\nMonthly raster aggregation complete.")
print(f"Outputs in: {OUTPUT_DIR}")


Processing MODIS EVI (16-day composites → monthly mean)...
  [EVI] 200101 — skipping (exists)
  [EVI] 200102 — skipping (exists)
  [EVI] 200103 — skipping (exists)
  [EVI] 200104 — skipping (exists)
  [EVI] 200105 — skipping (exists)
  [EVI] 200106 — skipping (exists)
  [EVI] 200107 — skipping (exists)
  [EVI] 200108 — skipping (exists)
  [EVI] 200109 — skipping (exists)
  [EVI] 200110 — skipping (exists)
  [EVI] 200111 — skipping (exists)
  [EVI] 200112 — skipping (exists)
  [EVI] 200201 — skipping (exists)
  [EVI] 200202 — skipping (exists)
  [EVI] 200203 — skipping (exists)
  [EVI] 200204 — skipping (exists)
  [EVI] 200205 — skipping (exists)
  [EVI] 200206 — skipping (exists)
  [EVI] 200207 — skipping (exists)
  [EVI] 200208 — skipping (exists)
  [EVI] 200209 — skipping (exists)
  [EVI] 200210 — skipping (exists)
  [EVI] 200211 — skipping (exists)
  [EVI] 200212 — skipping (exists)
  [EVI] 200301 — skipping (exists)
  [EVI] 200302 — skipping (exists)
  [EVI] 200303 — skipping (exi

In [6]:
# ── 4. MODIS LST (daily files, high missingness) ──────────────────────────────
#    Products: lst_d (day), lst_n (night), lst_diff (diurnal range)
#
#    Aggregates per product per month:
#       {product}_monthly_mean_YYYYMM.tif       — mean of available daily values
#       {product}_missing_fraction_YYYYMM.tif   — per-pixel fraction of days missing
#
#    Note on missing fraction: LST missingness is spatially structured (cloud
#    shadow, terrain). Unlike EVI (scalar missing fraction), LST missing fraction
#    is computed per pixel — mean of the NaN mask across the day stack.
#    A pixel with missing_fraction = 1.0 had no valid observations that month.
#
#    Note on lst_diff: if pre-computed daily files exist in the lst_diff
#    directory, they are processed identically. If that directory is absent,
#    set COMPUTE_LST_DIFF = True to derive it from lst_d minus lst_n stacks
#    for each month (only days with both d and n valid contribute).
 
print("\n" + "=" * 60)
print("Processing MODIS LST (daily, high missingness)...")
print("=" * 60)
 
LST_DIRS = {
    "lst_d":    r"C:\Users\A S U S\Documents\Study\ClimateData\CD-Workspace\Data\Datasets\MODIS_2_lst_d",
    "lst_n":    r"C:\Users\A S U S\Documents\Study\ClimateData\CD-Workspace\Data\Datasets\MODIS_2_lst_n",
    "lst_diff": r"C:\Users\A S U S\Documents\Study\ClimateData\CD-Workspace\Data\Datasets\MODIS_2_lst_diff",
}
 
# Set True if lst_diff daily files do not exist and should be computed from d - n
COMPUTE_LST_DIFF = False
 
lst_out = os.path.join(OUTPUT_DIR, "modis_lst")
os.makedirs(lst_out, exist_ok=True)
 
 
def stack_month_lst(catalog: pd.DataFrame, year: int, month: int):
    """
    Load LST daily rasters for year/month. Returns (stack, n_expected, n_available).
    stack is None if no files found.
    Missing fraction is computed per-pixel from NaN distribution in the stack,
    not as a scalar — LST cloud gaps are spatially structured.
    """
    month_files = catalog[
        (catalog["date"].dt.year  == year) &
        (catalog["date"].dt.month == month)
    ]
    days_in_month = pd.Period(f"{year}-{month:02d}", freq="M").days_in_month
 
    if month_files.empty:
        return None, days_in_month, 0
 
    slices = []
    for _, row in month_files.iterrows():
        da = rxr.open_rasterio(row["path"], masked=True)
        da = resample_to_template(da)
        if "band" in da.dims and da.sizes["band"] == 1:
            da = da.squeeze("band", drop=True)
        da = da.assign_coords(day=row["date"])
        slices.append(da.load())
        da.close()
 
    stack = xr.concat(slices, dim="day")
    return stack, days_in_month, len(slices)
 
 
def process_lst_product(product: str, directory: str, month_range):
    """Process one LST product directory → monthly mean + per-pixel missing fraction."""
    if not os.path.isdir(directory):
        print(f"  [{product}] Directory not found: {directory} — skipping")
        return
 
    catalog = filter_period(parse_tif_catalog(directory), START, END)
    if catalog.empty:
        print(f"  [{product}] No files found in period — skipping")
        return
 
    crs = rxr.open_rasterio(catalog["path"].iloc[0]).rio.crs
 
    for period in month_range:
        year, month = period.year, period.month
        label = month_label(year, month)
 
        mean_path    = os.path.join(lst_out, f"{product}_monthly_mean_{label}.tif")
        missing_path = os.path.join(lst_out, f"{product}_missing_fraction_{label}.tif")
 
        if os.path.exists(mean_path) and os.path.exists(missing_path):
            print(f"  [{product}] {label} — skipping (exists)")
            continue
 
        stack, n_expected, n_available = stack_month_lst(catalog, year, month)
 
        if stack is None:
            print(f"  [{product}] {label} — WARNING: no files found, skipping")
            continue
 
        # Monthly mean — skipna so valid pixels use whatever days are available
        monthly_mean = stack.mean(dim="day", skipna=True)
        save_raster(monthly_mean, mean_path, crs)
 
        # Per-pixel missing fraction:
        #   For each pixel, fraction of the month's calendar days with no valid value.
        #   Two sources of missingness: (a) file missing for that day entirely,
        #   (b) file present but pixel is NaN (cloud/QC mask).
        #
        #   We model (a) by treating entirely absent days as all-NaN planes,
        #   then missing_fraction = (NaN count per pixel) / days_in_month.
        nan_count_in_stack = stack.isnull().sum(dim="day")          # NaN days in stack
        absent_days        = n_expected - n_available                # days with no file at all
        total_nan_days     = nan_count_in_stack + absent_days        # per-pixel total
        missing_fraction   = (total_nan_days / n_expected).clip(0, 1)
        missing_fraction   = missing_fraction.astype(np.float32)
        save_raster(missing_fraction, missing_path, crs, nodata=-9999.0)
 
        del stack
        gc.collect()
 
        mean_missing = float(missing_fraction.mean())
        print(f"  [{product}] {label} — {n_available}/{n_expected} days, "
              f"mean missing fraction={mean_missing:.2f}")
 
 
# Process lst_d and lst_n unconditionally
for product in ["lst_d", "lst_n"]:
    process_lst_product(product, LST_DIRS[product], month_range)
 
# Process lst_diff: from files if available, otherwise derive from d - n stacks
if COMPUTE_LST_DIFF:
    print("\n  [lst_diff] Computing from lst_d minus lst_n stacks...")
 
    d_catalog = filter_period(parse_tif_catalog(LST_DIRS["lst_d"]), START, END)
    n_catalog = filter_period(parse_tif_catalog(LST_DIRS["lst_n"]), START, END)
    crs = rxr.open_rasterio(d_catalog["path"].iloc[0]).rio.crs
 
    for period in month_range:
        year, month = period.year, period.month
        label = month_label(year, month)
 
        mean_path    = os.path.join(lst_out, f"lst_diff_monthly_mean_{label}.tif")
        missing_path = os.path.join(lst_out, f"lst_diff_missing_fraction_{label}.tif")
 
        if os.path.exists(mean_path) and os.path.exists(missing_path):
            print(f"  [lst_diff] {label} — skipping (exists)")
            continue
 
        stack_d, n_exp_d, n_avail_d = stack_month_lst(d_catalog, year, month)
        stack_n, n_exp_n, n_avail_n = stack_month_lst(n_catalog, year, month)
 
        if stack_d is None or stack_n is None:
            print(f"  [lst_diff] {label} — WARNING: missing d or n stack, skipping")
            continue
 
        # Align on matching days only — inner join on day coordinate
        common_days = np.intersect1d(stack_d["day"].values, stack_n["day"].values)
        if len(common_days) == 0:
            print(f"  [lst_diff] {label} — WARNING: no matching d/n days, skipping")
            continue
 
        sd = stack_d.sel(day=common_days)
        sn = stack_n.sel(day=common_days)
        diff_stack = sd - sn   # NaN propagates if either d or n is missing for that day
 
        monthly_mean = diff_stack.mean(dim="day", skipna=True)
        save_raster(monthly_mean, mean_path, crs)
 
        nan_count      = diff_stack.isnull().sum(dim="day")
        absent_days    = n_exp_d - len(common_days)   # days absent in either product
        total_nan_days = nan_count + absent_days
        missing_frac   = (total_nan_days / n_exp_d).clip(0, 1).astype(np.float32)
        save_raster(missing_frac, missing_path, crs, nodata=-9999.0)
 
        del stack_d, stack_n, diff_stack
        gc.collect()
        print(f"  [lst_diff] {label} — {len(common_days)} matched days")
 
else:
    # lst_diff pre-computed files exist in their own directory
    process_lst_product("lst_diff", LST_DIRS["lst_diff"], month_range)
 


Processing MODIS LST (daily, high missingness)...
  [lst_d] 200101 — skipping (exists)
  [lst_d] 200102 — skipping (exists)
  [lst_d] 200103 — skipping (exists)
  [lst_d] 200104 — skipping (exists)
  [lst_d] 200105 — skipping (exists)
  [lst_d] 200106 — skipping (exists)
  [lst_d] 200107 — skipping (exists)
  [lst_d] 200108 — skipping (exists)
  [lst_d] 200109 — skipping (exists)
  [lst_d] 200110 — skipping (exists)
  [lst_d] 200111 — skipping (exists)
  [lst_d] 200112 — skipping (exists)
  [lst_d] 200201 — skipping (exists)
  [lst_d] 200202 — skipping (exists)
  [lst_d] 200203 — skipping (exists)
  [lst_d] 200204 — skipping (exists)
  [lst_d] 200205 — skipping (exists)
  [lst_d] 200206 — skipping (exists)
  [lst_d] 200207 — skipping (exists)
  [lst_d] 200208 — skipping (exists)
  [lst_d] 200209 — skipping (exists)
  [lst_d] 200210 — skipping (exists)
  [lst_d] 200211 — skipping (exists)
  [lst_d] 200212 — skipping (exists)
  [lst_d] 200301 — skipping (exists)
  [lst_d] 200302 — skipp